# Find the Fault (Grover's Search)

*Warm-up · UTC Problem 1*

One fault is hidden among `N` monitored nodes. Build Grover's search to find it in about √N steps instead of N.

**Your job:** fill in the two `TODO` functions (the oracle and the diffusion operator), run the search on a simulator for N = 4, 8, 16, then run it once on real hardware.

The Parts below match the README in this folder. Answer the written Part in your write-up.


In [ ]:
# qBraid Lab: install Qiskit Aer if this environment doesn't have it yet (one time, ~20 s)
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer

# Setup
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()


In [ ]:
# Problem parameters
N = 4                       # number of nodes (try 4, then 8, then 16)
n = int(np.log2(N))         # number of qubits
marked_node = 3             # which node is faulted (0..N-1); you can change this

# Optimal number of Grover iterations
iterations = int(np.floor((np.pi / 4) * np.sqrt(N)))
print(f"N={N}, qubits={n}, marked={marked_node}, iterations={iterations}")


In [ ]:
def oracle(qc, qubits, marked):
    """Flip the PHASE of the |marked> basis state, leaving all others unchanged."""
    # TODO: Implement the phase oracle.
    # Hint: put an X on each qubit that is 0 in the binary form of `marked`,
    #       apply a multi-controlled Z, then undo those X gates.
    #       A multi-controlled Z on `n` qubits = H on the target,
    #       mcx(controls, target), H on the target.
    raise NotImplementedError("Implement the oracle")


In [ ]:
def diffusion(qc, qubits):
    """Reflect the state about its average amplitude (the Grover diffuser)."""
    # TODO: Implement the diffusion operator.
    # Hint: H on all qubits, X on all qubits, a multi-controlled Z,
    #       X on all qubits, H on all qubits.
    raise NotImplementedError("Implement the diffusion operator")


In [ ]:
# Assemble the full Grover circuit (works once the TODOs are filled in)
def build_grover(n, marked, iterations):
    qc = QuantumCircuit(n, n)
    qubits = list(range(n))
    qc.h(qubits)                       # equal superposition
    for _ in range(iterations):
        oracle(qc, qubits, marked)
        diffusion(qc, qubits)
    qc.measure(qubits, qubits)
    return qc

qc = build_grover(n, marked_node, iterations)
qc.draw("mpl")


In [ ]:
# Run on the simulator
tqc = transpile(qc, sim)
counts = sim.run(tqc, shots=1024).result().get_counts()

# The measured node with the highest count should be the fault
best = max(counts, key=counts.get)
print("Most likely node (binary):", best, "= node", int(best, 2))

plt.bar(counts.keys(), counts.values())
plt.title(f"Grover search, N={N}")
plt.xlabel("measured node"); plt.ylabel("counts"); plt.show()


### Run it on a quantum computer through qBraid

Practice first on IonQ's simulator. It's free and can add realistic hardware noise. When your circuit works, switch the two lines to the real IonQ QPU for your hardware run. That run uses credits, so cap it at 200 shots and copy the job ID into your write-up.

In [ ]:
# --- Run on a quantum computer through qBraid ---
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()   # in qBraid Lab this uses your account automatically

# 1) Practice for free: IonQ's simulator with a realistic noise model
device = provider.get_device("ionq:ionq:sim:simulator")
job = device.run(qc, shots=200, runtime_options={"noise_model": "aria-1"})

# 2) Your real hardware run (uses credits). Swap these in when you're ready:
# device = provider.get_device("ionq:ionq:qpu:forte-1")
# job = device.run(qc, shots=200)

print("Job ID:", job.id)          # copy this into your write-up
hw_counts = job.result().data.get_counts()
print(hw_counts)

## What to submit
- **Part 1:** this notebook running correctly for N = 4, 8, 16, with a histogram peaking on the marked node each time.
- **Part 2:** the hardware histogram from the QPU cell.
- **Part 3:** a plot of checks vs N (classical vs Grover) and 2–3 sentences on the simulator-vs-hardware difference.
- **Part 4:** the written explanation, in your write-up.
